# **Assignment 10: Video Generation Diffusion- Trajectory Distribution Matching**

**Available:** Nov 4, 2025 3:00pm until Nov 13, 2025 11:59pm

**Details**
- https://huggingface.co/datasets/Gustavosta/Stable-Diffusion-Prompts/viewer/default/test?views%5B%5D=test
    - Go to the "People" section in the webcourse​
    - If you are the first student, for example, you will take the first 10 prompts from above link, second student will take the 11-20th prompts in the list and so on​
- Run SDXL Turbo and SDXL on those prompts assigned to you​
- Compare both models with two metrics​
    - FID​
    - Speed ​
- Run TDM on SDXL ​
    - Compare SDXL Turbo and TDM – FID and speed​
    - Report, video, code

## **Setup**

In [2]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Install all packages for from the requirements.txt
%pip install -r requirements.txt

import torch
import torch.nn as nn
import torchvision
import datasets
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline
import time
import psutil
import gc
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json
from collections import defaultdict
import numpy as np

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("Found Hugging Face token in environment variables")
    
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"Successfully authenticated with Hugging Face!")
        print(f"Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        print("Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("No Hugging Face token found in .env file")
    print("Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None


# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Found Hugging Face token in environment variables
Found Hugging Face token in environment variables


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully authenticated with Hugging Face!
Logged in as: malneyugnfl


In [3]:
# GPU Setup 
# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> mps (Apple Silicon) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    elif device.type == 'mps':
        # MPS doesn't have direct memory monitoring like CUDA
        # Return 0 as a placeholder
        return 0
    return 0


=== GPU Diagnostics ===
PyTorch version: 2.9.0+cu128
CUDA available: True
MPS available: False
CUDA version: 12.8
Number of GPUs detected: 2

=== All Available GPUs ===
GPU 0:
  Name: NVIDIA GeForce RTX 4090 Laptop GPU
  Total Memory: 15.70 GB
  Multi-processor count: 76
  Compute Capability: 8.9

GPU 1:
  Name: NVIDIA RTX A6000
  Total Memory: 47.53 GB
  Multi-processor count: 84
  Compute Capability: 8.6

Using GPU 1: NVIDIA RTX A6000
Selected device: cuda:1


In [4]:
# Function to clear memory for both CUDA and MPS
def clear_memory():
    """Clear CPU and GPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print(f"Cleared memory. Current CPU memory usage: {get_memory_usage():.2f} MB, GPU memory usage: {get_gpu_memory_usage():.2f} MB")

In [5]:
# Stable Diffusion Prompts:

prompt_list = [
    
"a girl smiling, Tristan Eaton, victo ngai, artgerm, RHADS, ross draws",
"a character portrait of a cyborg super saiyan in the style of moebius in the style of yoji shinkawa trending on artstation deviantart pinterest detailed realistic hd 8 k high resolution",
"big hand holding skull and bones, illustrated by Simon Stålenhag and Gaston Bussiere, intricate, ultra detailed, photorealistic, trending on artstation",
"Her huge ominous glowing blue eyes staring into my soul , perfect eyes, soft pale white skin, intricate stunning highly detailed, agostino arrivabene, artgerm, twisted dark lucid dream, 8k portrait render, raven angel wings, swirling thick smoke , beautiful lighting, dark fantasy art, cgsociety",
"the prince of frost standing alone in his court, archfey, full - body portrait, fantasy, white hair, blue skin, wild eyebrows, young adult, elf, crown, hard edges, soft lighting, professional lighting, trending on artstation",
"male cottagecore snoop dogg, Calvin Cordozar Broadus Jr., intricate, swagger, highly detailed, digital painting, artstation, concept art, smooth, sharp, focus, illustration, art by artgerm and greg rutkowski and alphonse mucha",
"amazing lifelike award winning pencil illustration of bob dylan trending on art station artgerm greg rutkowski alphonse mucha cinematic",
"medium - shot vislor turlough played by mark strickson at age 1 8, beautiful, at the alien space pub bar counter, highly detailed, mood lighting, artstation, highly detailed digital painting, smooth, global illumination, fantasy art by greg rutkowsky, karl spitzweg, leyendecker",
"portal to the multiverse of infinite horror with freakish spider creatures leaking out by laurie greasley digital art, highly detailed, intricate, sharp focus, trending on artstation hq, deviantart, octane render, bump map, pinterest, unreal engine 5, 4 k uhd image",
"concept art of fractal shaped spaceship by Michal Klimczak, high detail, artstation CG society"

]

# **Import SDXL and Run Prompts**

In [ ]:
# Import SDXL
# Source: https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0

from diffusers import DiffusionPipeline

with torch.no_grad():
    pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0")
    pipe.to(device)



In [ ]:
# Run Prompts

prompt = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"
image = pipe(prompt).images[0]

plt.imshow(image)
image.save("test_image.png")
plt.axis('off')  # Hide axes
plt.show()

logs = []

for idx,prompt in enumerate(prompt_list):
    print(f"Generating image for prompt: {prompt}")
    start_time = time.time()
    image = pipe(prompt=prompt).images[0]
    duration = time.time() - start_time

    # Save each image with a unique filename based on the prompt
    safe_prompt = "".join(c if c.isalnum() else "_" for c in prompt)[:50]  # Sanitize
    filename = f"{idx}_sdxl_{safe_prompt}.png"
    image.save(f"data/{filename}")
    print(f"Saved image to {filename} (generation time: {duration:.2f} sec)")
    
    # Display the image
    plt.imshow(image)
    plt.axis('off')  # Hide axes
    plt.show()

    # Log info
    logs.append({
        "idx": idx,
        "prompt": prompt,
        "filename": filename,
        "duration_sec": duration
    })

# Save logs to CSV
df_logs = pd.DataFrame(logs)
df_logs.to_csv("data/sdxl_base_inference_logs.csv", index=False)
print("Logs saved to data/sdxl_base_inference_logs.csv")

In [ ]:
# Delete Model
del pipe
clear_memory()

# **Import SDXL Turbo and Run Prompts**

In [ ]:
# Import SDXL Turbo
# Source: https://huggingface.co/stabilityai/sdxl-turbo

from diffusers import AutoPipelineForText2Image

with torch.no_grad():
    pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
    pipe.to(device)




In [ ]:
prompt = "A cinematic shot of a baby racoon wearing an intricate italian priest robe."


image = pipe(prompt=prompt).images[0]

image.save("sdxl_turbo_test_image.png")
plt.imshow(image)
plt.axis('off')  # Hide axes
plt.show()


In [ ]:
turbo_logs = []

for idx,prompt in enumerate(prompt_list):
    print(f"Generating image for prompt: {prompt}")
    start_time=time.time()
    image = pipe(prompt=prompt).images[0]
    duration = time.time() - start_time
    
    # Save each image with a unique filename based on the prompt
    safe_prompt = "".join(c if c.isalnum() else "_" for c in prompt)[:50]  # Sanitize
    filename = f"{idx}_sdxl_turbo_{safe_prompt}.png"
    image.save(f"data/{filename}")
    print(f"Saved image to {filename} (generation time: {duration:.2f} sec)")
    
    # Display the image
    plt.imshow(image)
    plt.axis('off')  # Hide axes
    plt.show()

        # Log info
    turbo_logs.append({
        "idx": idx,
        "prompt": prompt,
        "filename": filename,
        "duration_sec": duration
    })


# Save logs to a CSV file
df_logs = pd.DataFrame(turbo_logs)
df_logs.to_csv("logs/sdxl_turbo_inference_logs.csv", index=False)
print("Logs saved to logs/sdxl_turbo_inference_logs.csv")

In [ ]:
# Delete Model
del pipe
clear_memory()

## **Compare Generation Times and Do the FID Scores**

In [ ]:
# Compare Time from Logs

# Read the log files - both are in the logs directory
sdxl_base_logs = pd.read_csv("logs/sdxl_base_inference_logs.csv")
sdxl_turbo_logs = pd.read_csv("logs/sdxl_turbo_inference_logs.csv")

# Create a comparison table
comparison_data = []

for i in range(len(sdxl_base_logs)):
    base_row = sdxl_base_logs.iloc[i]
    turbo_row = sdxl_turbo_logs.iloc[i]
    
    comparison_data.append({
        'Prompt_ID': i,
        'Prompt': base_row['prompt'],
        'SDXL_Base_Time_sec': round(base_row['duration_sec'], 2),
        'SDXL_Turbo_Time_sec': round(turbo_row['duration_sec'], 2),
        'Speed_Improvement_Multiplier': round(base_row['duration_sec'] / turbo_row['duration_sec'], 2)
    })

# Create DataFrame for comparison
comparison_df = pd.DataFrame(comparison_data)

# Display the comparison table
print("=== Generation Time Comparison ===\n")
print(comparison_df.to_string(index=False))

# Calculate summary statistics
print(f"\n=== Summary Statistics ===")
print(f"SDXL Base - Average generation time: {comparison_df['SDXL_Base_Time_sec'].mean():.2f} seconds")
print(f"SDXL Base - Total generation time: {comparison_df['SDXL_Base_Time_sec'].sum():.2f} seconds")
print(f"SDXL Turbo - Average generation time: {comparison_df['SDXL_Turbo_Time_sec'].mean():.2f} seconds") 
print(f"SDXL Turbo - Total generation time: {comparison_df['SDXL_Turbo_Time_sec'].sum():.2f} seconds")
print(f"Average speed improvement: {comparison_df['Speed_Improvement_Multiplier'].mean():.2f}x faster")

# Save the comparison table
comparison_df.to_csv("logs/model_comparison_table.csv", index=False)
print(f"\nComparison table saved to logs/model_comparison_table.csv")

In [ ]:
# Install pytorch-fid for FID calculation
%pip install pytorch-fid

# Import necessary libraries for FID calculation
import os
import glob
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from scipy import linalg
from torch.utils.data import Dataset, DataLoader

# Custom dataset for loading images
class ImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image

# Load pre-trained Inception v3 model for feature extraction
def load_inception_model(device):
    """Load and prepare Inception v3 model for FID calculation"""
    inception = models.inception_v3(pretrained=True, transform_input=False)
    inception.fc = nn.Identity()  # Remove final classification layer
    inception.eval()
    inception.to(device)
    return inception

# Extract features using Inception v3
def extract_features(model, dataloader, device):
    """Extract features from images using Inception v3"""
    features = []
    model.eval()
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting features"):
            batch = batch.to(device)
            # Inception v3 expects 299x299 images
            if batch.shape[-1] != 299:
                batch = F.interpolate(batch, size=(299, 299), mode='bilinear', align_corners=False)
            
            feat = model(batch)
            features.append(feat.cpu().numpy())
    
    return np.concatenate(features, axis=0)

# Calculate FID score
def calculate_fid(features1, features2):
    """Calculate FID score between two sets of features"""
    # Calculate mean and covariance for both feature sets
    mu1, sigma1 = features1.mean(axis=0), np.cov(features1, rowvar=False)
    mu2, sigma2 = features2.mean(axis=0), np.cov(features2, rowvar=False)
    
    # Calculate sum squared difference between means
    ssdiff = np.sum((mu1 - mu2) ** 2.0)
    
    # Calculate sqrt of product between covariances
    covmean = linalg.sqrtm(sigma1.dot(sigma2))
    
    # Check for imaginary numbers and take real part
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    # Calculate FID
    fid = ssdiff + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return fid

# Prepare data transforms for Inception v3
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Setting up FID calculation...")

# Get image paths for both models
data_dir = "data"
sdxl_base_images = sorted(glob.glob(os.path.join(data_dir, "*_sdxl_*.png")))
sdxl_turbo_images = sorted(glob.glob(os.path.join(data_dir, "*_sdxl_turbo_*.png")))

print(f"Found {len(sdxl_base_images)} SDXL Base images")
print(f"Found {len(sdxl_turbo_images)} SDXL Turbo images")

# Create datasets and dataloaders
base_dataset = ImageDataset(sdxl_base_images, transform=transform)
turbo_dataset = ImageDataset(sdxl_turbo_images, transform=transform)

base_dataloader = DataLoader(base_dataset, batch_size=4, shuffle=False)
turbo_dataloader = DataLoader(turbo_dataset, batch_size=4, shuffle=False)

# Load Inception model
print("Loading Inception v3 model...")
inception_model = load_inception_model(device)

# Extract features for both sets of images
print("Extracting features from SDXL Base images...")
base_features = extract_features(inception_model, base_dataloader, device)

print("Extracting features from SDXL Turbo images...")
turbo_features = extract_features(inception_model, turbo_dataloader, device)

# Calculate FID score
print("Calculating FID score...")
fid_score = calculate_fid(base_features, turbo_features)

print(f"\n=== FID Score Results ===")
print(f"FID Score between SDXL Base and SDXL Turbo: {fid_score:.4f}")

# Lower FID scores indicate better quality and more similar distributions
# Typically, FID scores:
# - 0-10: Excellent quality, very similar to reference
# - 10-25: Good quality  
# - 25-50: Moderate quality
# - 50+: Poor quality

if fid_score < 10:
    quality_assessment = "Excellent - Very similar image quality and distribution"
elif fid_score < 25:
    quality_assessment = "Good - Similar image quality with some differences"
elif fid_score < 50:
    quality_assessment = "Moderate - Noticeable differences in image quality"
else:
    quality_assessment = "Poor - Significant differences in image quality"

print(f"Quality Assessment: {quality_assessment}")

# Save FID results
fid_results = {
    'SDXL_Base_Images': len(sdxl_base_images),
    'SDXL_Turbo_Images': len(sdxl_turbo_images),
    'FID_Score': fid_score,
    'Quality_Assessment': quality_assessment
}

import json
with open('logs/fid_results.json', 'w') as f:
    json.dump(fid_results, f, indent=2)

print(f"\nFID results saved to logs/fid_results.json")

# Clean up memory
del inception_model
clear_memory()

In [ ]:
# Comprehensive Model Comparison Analysis
print("=== COMPREHENSIVE MODEL COMPARISON ===\n")

# Load timing comparison data
timing_df = pd.read_csv("logs/model_comparison_table.csv")

# Load FID results
with open('logs/fid_results.json', 'r') as f:
    fid_data = json.load(f)

print("1. SPEED COMPARISON:")
print(f"   • SDXL Base average time: {timing_df['SDXL_Base_Time_sec'].mean():.2f} seconds")
print(f"   • SDXL Turbo average time: {timing_df['SDXL_Turbo_Time_sec'].mean():.2f} seconds") 
print(f"   • Speed improvement: {timing_df['Speed_Improvement_Multiplier'].mean():.2f}x faster\n")

print("2. IMAGE QUALITY COMPARISON (FID Score):")
print(f"   • FID Score: {fid_data['FID_Score']:.4f}")
print(f"   • Assessment: {fid_data['Quality_Assessment']}\n")

print("3. TRADE-OFF ANALYSIS:")
speed_improvement = timing_df['Speed_Improvement_Multiplier'].mean()
fid_score = fid_data['FID_Score']

if fid_score < 25 and speed_improvement > 5:
    trade_off = "Excellent trade-off: Significant speed gain with minimal quality loss"
elif fid_score < 50 and speed_improvement > 3:
    trade_off = "Good trade-off: Good speed improvement with acceptable quality difference"
elif fid_score < 100:
    trade_off = "Moderate trade-off: Speed improvement comes with noticeable quality reduction"
else:
    trade_off = "Poor trade-off: Speed improvement comes with significant quality loss"

print(f"   • {trade_off}")

print("\n4. SUMMARY:")
print(f"   SDXL Turbo generates images {speed_improvement:.1f}x faster than SDXL Base")
print(f"   with an FID score of {fid_score:.2f}, indicating {fid_data['Quality_Assessment'].lower()}")

# Create a summary dataframe for easy reference
summary_data = {
    'Metric': ['Average Generation Time (SDXL Base)', 'Average Generation Time (SDXL Turbo)', 
               'Speed Improvement', 'FID Score', 'Quality Assessment'],
    'Value': [f"{timing_df['SDXL_Base_Time_sec'].mean():.2f} sec", 
              f"{timing_df['SDXL_Turbo_Time_sec'].mean():.2f} sec",
              f"{speed_improvement:.2f}x", 
              f"{fid_score:.4f}",
              fid_data['Quality_Assessment']]
}

summary_df = pd.DataFrame(summary_data)
print(f"\n=== SUMMARY TABLE ===")
print(summary_df.to_string(index=False))

# Save comprehensive results
summary_df.to_csv("logs/comprehensive_model_comparison.csv", index=False)
print(f"\nComprehensive comparison saved to logs/comprehensive_model_comparison.csv")

## Implement Trajectory Distribution Matching (TDM) on SDXL

### References:
- TDM Paper: https://arxiv.org/abs/2410.01739
- Original TDM source: https://huggingface.co/Luo-Yihong/TDM
- SDXL: https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0

In [ ]:
# Because the weights for TDM-LoRA on SDXL are not available, we will demonstrate TDM on Stable Diffusion 3 instead.

import torch
from diffusers import StableDiffusion3Pipeline, AutoencoderTiny, DPMSolverMultistepScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from diffusers.utils import make_image_grid
tdm_pipe = StableDiffusion3Pipeline.from_pretrained("stabilityai/stable-diffusion-3-medium-diffusers", torch_dtype=torch.float16).to(device)
tdm_pipe.load_lora_weights('Luo-Yihong/TDM_sd3_lora', adapter_name = 'tdm') # Load TDM-LoRA
tdm_pipe.set_adapters(["tdm"], [0.125])# IMPORTANT. Please set LoRA scale to 0.125.
tdm_pipe.vae = AutoencoderTiny.from_pretrained("madebyollin/taesd3", torch_dtype=torch.float16) # Save GPU memory.
tdm_pipe.vae.config.shift_factor = 0.0
tdm_pipe = tdm_pipe.to(device)


tdm_pipe.scheduler = DPMSolverMultistepScheduler.from_pretrained("Efficient-Large-Model/Sana_1600M_1024px_BF16_diffusers", subfolder="scheduler")
tdm_pipe.scheduler.config['flow_shift'] = 6 # the flow_shift can be changed from 1 to 6.
tdm_pipe.scheduler = DPMSolverMultistepScheduler.from_config(tdm_pipe.scheduler.config)
generator = torch.manual_seed(8888)
image = tdm_pipe(
    prompt="A cute panda holding a sign says TDM SOTA!",
    negative_prompt="",
    num_inference_steps=4,
    height=1024,
    width=1024,
    num_images_per_prompt = 1,
    guidance_scale=1.,
    generator = generator,
).images[0]



tdm_pipe.scheduler = DPMSolverMultistepScheduler.from_pretrained("Efficient-Large-Model/Sana_1600M_1024px_BF16_diffusers", subfolder="scheduler")
tdm_pipe.set_adapters(["tdm"], [0.]) # Unload lora
generator = torch.manual_seed(8888)
teacher_image = tdm_pipe(
    prompt="A cute panda holding a sign says TDM SOTA!",
    negative_prompt="",
    num_inference_steps=28,
    height=1024,
    width=1024,
    num_images_per_prompt = 1,
    guidance_scale=7.,
    generator = generator,
).images[0]
make_image_grid([image,teacher_image],1,2)


In [ ]:
# Loop through prompt list and generate images with TDM

tdm_logs = []

for idx,prompt in enumerate(prompt_list):
    print(f"Generating image for prompt: {prompt}")
    start_time=time.time()
    image = tdm_pipe(prompt=prompt).images[0]
    duration = time.time() - start_time
    
    # Save each image with a unique filename based on the prompt
    safe_prompt = "".join(c if c.isalnum() else "_" for c in prompt)[:50]  # Sanitize
    filename = f"{idx}_tdm_{safe_prompt}.png"
    image.save(f"data/{filename}")
    print(f"Saved image to {filename} (generation time: {duration:.2f} sec)")
    
    # Display the image
    plt.imshow(image)
    plt.axis('off')  # Hide axes
    plt.show()

        # Log info
    tdm_logs.append({
        "idx": idx,
        "prompt": prompt,
        "filename": filename,
        "duration_sec": duration
    })


# Save logs to a CSV file
df_logs = pd.DataFrame(tdm_logs)
df_logs.to_csv("logs/tdm_inference_logs.csv", index=False)
print("Logs saved to logs/tdm_inference_logs.csv")

## Model Comparisons

In [6]:
# Install pytorch-fid for FID calculation
%pip install pytorch-fid

# Import necessary libraries for FID calculation
import os
import glob
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from scipy import linalg
from torch.utils.data import Dataset, DataLoader

# Custom dataset for loading images
class ImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image

# Load pre-trained Inception v3 model for feature extraction
def load_inception_model(device):
    """Load and prepare Inception v3 model for FID calculation"""
    inception = models.inception_v3(pretrained=True, transform_input=False)
    inception.fc = nn.Identity()  # Remove final classification layer
    inception.eval()
    inception.to(device)
    return inception

# Extract features using Inception v3
def extract_features(model, dataloader, device):
    """Extract features from images using Inception v3"""
    features = []
    model.eval()
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting features"):
            batch = batch.to(device)
            # Inception v3 expects 299x299 images
            if batch.shape[-1] != 299:
                batch = F.interpolate(batch, size=(299, 299), mode='bilinear', align_corners=False)
            
            feat = model(batch)
            features.append(feat.cpu().numpy())
    
    return np.concatenate(features, axis=0)

# Calculate FID score
def calculate_fid(features1, features2):
    """Calculate FID score between two sets of features"""
    # Calculate mean and covariance for both feature sets
    mu1, sigma1 = features1.mean(axis=0), np.cov(features1, rowvar=False)
    mu2, sigma2 = features2.mean(axis=0), np.cov(features2, rowvar=False)
    
    # Calculate sum squared difference between means
    ssdiff = np.sum((mu1 - mu2) ** 2.0)
    
    # Calculate sqrt of product between covariances
    covmean = linalg.sqrtm(sigma1.dot(sigma2))
    
    # Check for imaginary numbers and take real part
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    # Calculate FID
    fid = ssdiff + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return fid

# Prepare data transforms for Inception v3
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Setting up FID calculation...")

# Get image paths for all three models
data_dir = "data"
sdxl_base_images = sorted(glob.glob(os.path.join(data_dir, "*_sdxl_*.png")))
sdxl_turbo_images = sorted(glob.glob(os.path.join(data_dir, "*_sdxl_turbo_*.png")))
tdm_images = sorted(glob.glob(os.path.join(data_dir, "*_tdm_*.png")))

print(f"Found {len(sdxl_base_images)} SDXL Base images")
print(f"Found {len(sdxl_turbo_images)} SDXL Turbo images")
print(f"Found {len(tdm_images)} TDM images")

# Create datasets and dataloaders for all three models
base_dataset = ImageDataset(sdxl_base_images, transform=transform)
turbo_dataset = ImageDataset(sdxl_turbo_images, transform=transform)
tdm_dataset = ImageDataset(tdm_images, transform=transform)

base_dataloader = DataLoader(base_dataset, batch_size=4, shuffle=False)
turbo_dataloader = DataLoader(turbo_dataset, batch_size=4, shuffle=False)
tdm_dataloader = DataLoader(tdm_dataset, batch_size=4, shuffle=False)

# Load Inception model
print("Loading Inception v3 model...")
inception_model = load_inception_model(device)

# Extract features for all three sets of images
print("Extracting features from SDXL Base images...")
base_features = extract_features(inception_model, base_dataloader, device)

print("Extracting features from SDXL Turbo images...")
turbo_features = extract_features(inception_model, turbo_dataloader, device)

print("Extracting features from TDM images...")
tdm_features = extract_features(inception_model, tdm_dataloader, device)

# Calculate all pairwise FID scores
print("\nCalculating FID scores...")
fid_base_turbo = calculate_fid(base_features, turbo_features)
fid_base_tdm = calculate_fid(base_features, tdm_features)
fid_turbo_tdm = calculate_fid(turbo_features, tdm_features)

print(f"\n=== FID Score Results ===")
print(f"FID Score between SDXL Base and SDXL Turbo: {fid_base_turbo:.4f}")
print(f"FID Score between SDXL Base and TDM: {fid_base_tdm:.4f}")
print(f"FID Score between SDXL Turbo and TDM: {fid_turbo_tdm:.4f}")

# Function to assess quality based on FID score
def assess_quality(fid_score):
    """Assess image quality based on FID score"""
    # Lower FID scores indicate better quality and more similar distributions
    # Typically, FID scores:
    # - 0-10: Excellent quality, very similar to reference
    # - 10-25: Good quality  
    # - 25-50: Moderate quality
    # - 50+: Poor quality
    if fid_score < 10:
        return "Excellent - Very similar image quality and distribution"
    elif fid_score < 25:
        return "Good - Similar image quality with some differences"
    elif fid_score < 50:
        return "Moderate - Noticeable differences in image quality"
    else:
        return "Poor - Significant differences in image quality"

print("\n=== Quality Assessments ===")
print(f"SDXL Base vs SDXL Turbo: {assess_quality(fid_base_turbo)}")
print(f"SDXL Base vs TDM: {assess_quality(fid_base_tdm)}")
print(f"SDXL Turbo vs TDM: {assess_quality(fid_turbo_tdm)}")

# Save comprehensive FID results
fid_results = {
    'SDXL_Base_Images': len(sdxl_base_images),
    'SDXL_Turbo_Images': len(sdxl_turbo_images),
    'TDM_Images': len(tdm_images),
    'FID_Base_vs_Turbo': float(fid_base_turbo),
    'FID_Base_vs_TDM': float(fid_base_tdm),
    'FID_Turbo_vs_TDM': float(fid_turbo_tdm),
    'Quality_Assessment_Base_vs_Turbo': assess_quality(fid_base_turbo),
    'Quality_Assessment_Base_vs_TDM': assess_quality(fid_base_tdm),
    'Quality_Assessment_Turbo_vs_TDM': assess_quality(fid_turbo_tdm)
}

import json
with open('logs/fid_results_comprehensive.json', 'w') as f:
    json.dump(fid_results, f, indent=2)

print(f"\nComprehensive FID results saved to logs/fid_results_comprehensive.json")

# Create a comparison table
comparison_data = {
    'Comparison': ['SDXL Base vs SDXL Turbo', 'SDXL Base vs TDM', 'SDXL Turbo vs TDM'],
    'FID_Score': [f"{fid_base_turbo:.4f}", f"{fid_base_tdm:.4f}", f"{fid_turbo_tdm:.4f}"],
    'Quality_Assessment': [
        assess_quality(fid_base_turbo),
        assess_quality(fid_base_tdm),
        assess_quality(fid_turbo_tdm)
    ]
}

fid_comparison_df = pd.DataFrame(comparison_data)
print(f"\n=== FID Comparison Table ===")
print(fid_comparison_df.to_string(index=False))

# Save comparison table
fid_comparison_df.to_csv("logs/fid_comparison_table.csv", index=False)
print(f"\nFID comparison table saved to logs/fid_comparison_table.csv")

# Clean up memory
del inception_model
clear_memory()


Note: you may need to restart the kernel to use updated packages.
Setting up FID calculation...
Found 20 SDXL Base images
Found 10 SDXL Turbo images
Found 10 TDM images
Loading Inception v3 model...
Note: you may need to restart the kernel to use updated packages.
Setting up FID calculation...
Found 20 SDXL Base images
Found 10 SDXL Turbo images
Found 10 TDM images
Loading Inception v3 model...
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /home/malneyugnfl/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /home/malneyugnfl/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 104M/104M [00:01<00:00, 55.4MB/s] 



Extracting features from SDXL Base images...


Extracting features:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting features from SDXL Turbo images...


Extracting features:   0%|          | 0/3 [00:00<?, ?it/s]

Extracting features from TDM images...


Extracting features:   0%|          | 0/3 [00:00<?, ?it/s]


Calculating FID scores...

=== FID Score Results ===
FID Score between SDXL Base and SDXL Turbo: 114.7479
FID Score between SDXL Base and TDM: 263.7083
FID Score between SDXL Turbo and TDM: 289.3972

=== Quality Assessments ===
SDXL Base vs SDXL Turbo: Poor - Significant differences in image quality
SDXL Base vs TDM: Poor - Significant differences in image quality
SDXL Turbo vs TDM: Poor - Significant differences in image quality

Comprehensive FID results saved to logs/fid_results_comprehensive.json

=== FID Comparison Table ===
             Comparison FID_Score                              Quality_Assessment
SDXL Base vs SDXL Turbo  114.7479 Poor - Significant differences in image quality
       SDXL Base vs TDM  263.7083 Poor - Significant differences in image quality
      SDXL Turbo vs TDM  289.3972 Poor - Significant differences in image quality

FID comparison table saved to logs/fid_comparison_table.csv

=== FID Score Results ===
FID Score between SDXL Base and SDXL Turbo: 114

In [7]:
# Comprehensive Model Comparison Analysis (All Three Models)
print("=== COMPREHENSIVE MODEL COMPARISON (SDXL Base, SDXL Turbo, and TDM) ===\n")

# Load timing comparison data
timing_df_base_turbo = pd.read_csv("logs/model_comparison_table.csv")
timing_df_tdm = pd.read_csv("logs/tdm_inference_logs.csv")

# Load FID results
with open('logs/fid_results_comprehensive.json', 'r') as f:
    fid_data = json.load(f)

print("1. SPEED COMPARISON:")
print(f"   • SDXL Base average time: {timing_df_base_turbo['SDXL_Base_Time_sec'].mean():.2f} seconds")
print(f"   • SDXL Turbo average time: {timing_df_base_turbo['SDXL_Turbo_Time_sec'].mean():.2f} seconds")
print(f"   • TDM average time: {timing_df_tdm['duration_sec'].mean():.2f} seconds")
print(f"   • SDXL Turbo vs Base speed improvement: {timing_df_base_turbo['Speed_Improvement_Multiplier'].mean():.2f}x faster")
print(f"   • TDM vs Base speed improvement: {timing_df_base_turbo['SDXL_Base_Time_sec'].mean() / timing_df_tdm['duration_sec'].mean():.2f}x faster")
print(f"   • TDM vs Turbo speed comparison: {timing_df_base_turbo['SDXL_Turbo_Time_sec'].mean() / timing_df_tdm['duration_sec'].mean():.2f}x faster\n")

print("2. IMAGE QUALITY COMPARISON (FID Scores):")
print(f"   • FID Score (SDXL Base vs SDXL Turbo): {fid_data['FID_Base_vs_Turbo']:.4f}")
print(f"     Assessment: {fid_data['Quality_Assessment_Base_vs_Turbo']}")
print(f"   • FID Score (SDXL Base vs TDM): {fid_data['FID_Base_vs_TDM']:.4f}")
print(f"     Assessment: {fid_data['Quality_Assessment_Base_vs_TDM']}")
print(f"   • FID Score (SDXL Turbo vs TDM): {fid_data['FID_Turbo_vs_TDM']:.4f}")
print(f"     Assessment: {fid_data['Quality_Assessment_Turbo_vs_TDM']}\n")

print("3. TRADE-OFF ANALYSIS:")

# SDXL Turbo vs Base
speed_improvement_turbo = timing_df_base_turbo['Speed_Improvement_Multiplier'].mean()
fid_score_base_turbo = fid_data['FID_Base_vs_Turbo']

if fid_score_base_turbo < 25 and speed_improvement_turbo > 5:
    trade_off_turbo = "Excellent trade-off: Significant speed gain with minimal quality loss"
elif fid_score_base_turbo < 50 and speed_improvement_turbo > 3:
    trade_off_turbo = "Good trade-off: Good speed improvement with acceptable quality difference"
elif fid_score_base_turbo < 100:
    trade_off_turbo = "Moderate trade-off: Speed improvement comes with noticeable quality reduction"
else:
    trade_off_turbo = "Poor trade-off: Speed improvement comes with significant quality loss"

print(f"   SDXL Turbo vs Base: {trade_off_turbo}")

# TDM vs Base
speed_improvement_tdm = timing_df_base_turbo['SDXL_Base_Time_sec'].mean() / timing_df_tdm['duration_sec'].mean()
fid_score_base_tdm = fid_data['FID_Base_vs_TDM']

if fid_score_base_tdm < 25 and speed_improvement_tdm > 5:
    trade_off_tdm_base = "Excellent trade-off: Significant speed gain with minimal quality loss"
elif fid_score_base_tdm < 50 and speed_improvement_tdm > 3:
    trade_off_tdm_base = "Good trade-off: Good speed improvement with acceptable quality difference"
elif fid_score_base_tdm < 100:
    trade_off_tdm_base = "Moderate trade-off: Speed improvement comes with noticeable quality reduction"
else:
    trade_off_tdm_base = "Poor trade-off: Speed improvement comes with significant quality loss"

print(f"   TDM vs SDXL Base: {trade_off_tdm_base}")

# TDM vs Turbo
speed_comparison_tdm_turbo = timing_df_base_turbo['SDXL_Turbo_Time_sec'].mean() / timing_df_tdm['duration_sec'].mean()
fid_score_turbo_tdm = fid_data['FID_Turbo_vs_TDM']

if fid_score_turbo_tdm < 25:
    trade_off_tdm_turbo = f"TDM is {speed_comparison_tdm_turbo:.2f}x the speed of Turbo with similar quality (FID: {fid_score_turbo_tdm:.2f})"
elif fid_score_turbo_tdm < 50:
    trade_off_tdm_turbo = f"TDM is {speed_comparison_tdm_turbo:.2f}x the speed of Turbo with moderate quality difference (FID: {fid_score_turbo_tdm:.2f})"
else:
    trade_off_tdm_turbo = f"TDM is {speed_comparison_tdm_turbo:.2f}x the speed of Turbo with noticeable quality difference (FID: {fid_score_turbo_tdm:.2f})"

print(f"   TDM vs SDXL Turbo: {trade_off_tdm_turbo}\n")

print("4. SUMMARY:")
print(f"   • SDXL Turbo generates images {speed_improvement_turbo:.1f}x faster than SDXL Base")
print(f"     with an FID score of {fid_score_base_turbo:.2f}")
print(f"   • TDM generates images {speed_improvement_tdm:.1f}x faster than SDXL Base")
print(f"     with an FID score of {fid_score_base_tdm:.2f}")
print(f"   • Comparing TDM vs SDXL Turbo: TDM is {speed_comparison_tdm_turbo:.2f}x the speed")
print(f"     with an FID score of {fid_score_turbo_tdm:.2f}\n")

# Create a comprehensive summary dataframe
summary_data = {
    'Metric': [
        'Avg Generation Time (SDXL Base)', 
        'Avg Generation Time (SDXL Turbo)', 
        'Avg Generation Time (TDM)',
        'Speed: Turbo vs Base', 
        'Speed: TDM vs Base',
        'Speed: TDM vs Turbo',
        'FID: Base vs Turbo', 
        'FID: Base vs TDM',
        'FID: Turbo vs TDM'
    ],
    'Value': [
        f"{timing_df_base_turbo['SDXL_Base_Time_sec'].mean():.2f} sec", 
        f"{timing_df_base_turbo['SDXL_Turbo_Time_sec'].mean():.2f} sec",
        f"{timing_df_tdm['duration_sec'].mean():.2f} sec",
        f"{speed_improvement_turbo:.2f}x faster",
        f"{speed_improvement_tdm:.2f}x faster",
        f"{speed_comparison_tdm_turbo:.2f}x",
        f"{fid_score_base_turbo:.4f}",
        f"{fid_score_base_tdm:.4f}",
        f"{fid_score_turbo_tdm:.4f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("=== SUMMARY TABLE ===")
print(summary_df.to_string(index=False))

# Save comprehensive results
summary_df.to_csv("logs/comprehensive_model_comparison_all.csv", index=False)
print(f"\nComprehensive comparison saved to logs/comprehensive_model_comparison_all.csv")


=== COMPREHENSIVE MODEL COMPARISON (SDXL Base, SDXL Turbo, and TDM) ===

1. SPEED COMPARISON:
   • SDXL Base average time: 40.60 seconds
   • SDXL Turbo average time: 2.95 seconds
   • TDM average time: 8.50 seconds
   • SDXL Turbo vs Base speed improvement: 13.77x faster
   • TDM vs Base speed improvement: 4.78x faster
   • TDM vs Turbo speed comparison: 0.35x faster

2. IMAGE QUALITY COMPARISON (FID Scores):
   • FID Score (SDXL Base vs SDXL Turbo): 114.7479
     Assessment: Poor - Significant differences in image quality
   • FID Score (SDXL Base vs TDM): 263.7083
     Assessment: Poor - Significant differences in image quality
   • FID Score (SDXL Turbo vs TDM): 289.3972
     Assessment: Poor - Significant differences in image quality

3. TRADE-OFF ANALYSIS:
   SDXL Turbo vs Base: Poor trade-off: Speed improvement comes with significant quality loss
   TDM vs SDXL Base: Poor trade-off: Speed improvement comes with significant quality loss
   TDM vs SDXL Turbo: TDM is 0.35x the spee